# Quantum Teleportation

**Learning objectives**

- Build the teleportation protocol
- Use Bell measurement and conditional correction
- Verify arbitrary-state fidelity

## Environment setup



In [ ]:
# Run once in a fresh Colab/Jupyter environment, then restart the kernel if prompted.
%pip install -q qiskit qiskit-aer matplotlib pylatexenc scipy sympy

In [ ]:
from importlib.metadata import version
for package in ["qiskit", "qiskit-aer", "matplotlib", "pylatexenc", "scipy", "sympy"]:
    print(f"{package:15}: {version(package)}")

## Coherent teleportation circuit

Deferred measurement replaces classical feed-forward with coherent controlled operations, enabling clean local verification.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, partial_trace, state_fidelity
theta,phi=1.1,0.7
qc=QuantumCircuit(3)
qc.ry(theta,0); qc.rz(phi,0)          # unknown state
target=Statevector.from_instruction(QuantumCircuit(1).compose(QuantumCircuit(1)))
prep=QuantumCircuit(1); prep.ry(theta,0); prep.rz(phi,0); target=Statevector.from_instruction(prep)
qc.h(1); qc.cx(1,2)                   # entangled channel
qc.cx(0,1); qc.h(0)                   # Bell-basis transform
qc.cx(1,2); qc.cz(0,2)                # deferred conditional corrections
final=Statevector.from_instruction(qc)
bob=partial_trace(final,[0,1])
display(qc.draw("mpl")); display(target.draw("latex"))
print("Bob density matrix:\n",bob.data); print("Teleportation fidelity:",state_fidelity(bob,target))

## Test several input states

A correct protocol gives fidelity one for each input.

In [ ]:
for theta,phi in [(0,0),(np.pi,0),(np.pi/2,0),(1.2,2.1)]:
    q=QuantumCircuit(3); q.ry(theta,0); q.rz(phi,0); p=QuantumCircuit(1); p.ry(theta,0); p.rz(phi,0)
    q.h(1); q.cx(1,2); q.cx(0,1); q.h(0); q.cx(1,2); q.cz(0,2)
    print(theta,phi,state_fidelity(partial_trace(Statevector.from_instruction(q),[0,1]),Statevector.from_instruction(p)))